# Notebook 04: Speech Recognition (ASR)

**Time:** 25 minutes  
**Prerequisites:** Notebook 03 complete  
**Goal:** Transcribe audio using Whisper and faster-whisper, compare speed and accuracy

This notebook will:
1. Transcribe audio with OpenAI Whisper (baseline)
2. Use faster-whisper for 4x speedup with CTranslate2
3. Compare model sizes and their speed/accuracy trade-offs
4. Discuss real-world ASR for pretraining data (podcasts, lectures, YouTube)

> **Why this matters:** Audio is a massive untapped data source for LLM pretraining. Podcasts, lectures, and YouTube videos contain billions of tokens of high-quality spoken content. ASR converts this audio to text. Whisper v3 turbo (2025) processes audio 216x faster than real-time.

In [1]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

import src.audio_utils
importlib.reload(src.audio_utils)
from src.audio_utils import (
    transcribe_with_faster_whisper,
)

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 04")

✓ Ollama client initialized
  Available models: ['nomic-embed-text:latest', 'qwen3.5:latest', 'gemma4:e2b', 'llama3:latest', 'granite4:latest']
  Default model: granite4:latest
Setup complete -- ready for Notebook 04


---

## Part 1: OpenAI Whisper (Baseline)

Whisper is OpenAI's open-source ASR model. It supports 100+ languages and multiple model sizes:

| Model | Params | Speed | Best For |
|-------|--------|-------|----------|
| tiny | 39M | Fastest | Quick testing |
| base | 74M | Fast | Good default |
| small | 244M | Medium | Better accuracy |
| medium | 769M | Slow | High accuracy |
| large-v3 | 1.5B | Slowest | Best accuracy |
| turbo | 809M | 6x faster than large | Best speed/accuracy (2025) |

In [2]:
print("=" * 65)
print("Experiment 1: OpenAI Whisper Transcription")
print("=" * 65)
print()

# Find test audio
test_audio = os.path.join(parent_dir, 'test_data', 'audio', 'sample-1.mp3')

if not os.path.exists(test_audio):
    # Try Class3 test data
    class3_audio = os.path.join(parent_dir, '..', 'MLE_in_Gen_AI-Course', 'Class3', 'test_data', 'audio', 'sample-1.mp3')
    if os.path.exists(class3_audio):
        test_audio = class3_audio
    else:
        print("No test audio found.")
        print("Place an audio file at: test_data/audio/sample-1.mp3")
        test_audio = None

if test_audio:
    try:
        from src.audio_utils import transcribe_with_whisper
        whisper_result = transcribe_with_whisper(test_audio, model_size="base")
    except ImportError as e:
        print(f"Whisper not installed: {e}")
        print("Install with: pip install openai-whisper")
        print("Continuing with faster-whisper in Part 2...")

Experiment 1: OpenAI Whisper Transcription

WHISPER ASR (model: base)
Audio: /home/conardd/working_space/Homework3-Submission/test_data/audio/sample-1.mp3


/home/conardd/miniconda3/envs/hk2/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


  Language: en
  Segments: 7
  Transcribed in 7.2s
  Text: So we pace that, they didn't, you know, it is some people's success on grand, them and so instead of get an musical. I just notレ the...


---

## Part 2: faster-whisper (4x Speedup)

**faster-whisper** uses CTranslate2 to run Whisper models 4x faster with lower memory. It's the recommended backend for production ASR.

Key features:
- Supports all Whisper model sizes including **turbo**
- Batched inference for processing multiple files
- INT8 quantization for CPU efficiency
- Word-level timestamps

In [3]:
print("=" * 65)
print("Experiment 2: faster-whisper Transcription")
print("=" * 65)
print()

if test_audio:
    fw_result = transcribe_with_faster_whisper(
        test_audio,
        model_size="base",
        device="cpu",
        compute_type="int8"
    )
    
    print("\nTimestamped segments:")
    for seg in fw_result['segments']:
        print(f"  [{seg['start']:.1f}s - {seg['end']:.1f}s] {seg['text']}")
else:
    print("No test audio available.")

Experiment 2: faster-whisper Transcription

FASTER-WHISPER ASR (model: base, cpu/int8)
Audio: /home/conardd/working_space/Homework3-Submission/test_data/audio/sample-1.mp3
  Language: en (prob: 0.687)
  Segments: 4
  Transcribed in 2.9s
  Text: So we pay for that, they didn't, you know, there are some people who said, it's on Kren the Benzo instead of Quotan and musical, or there's not a story of what you're interested in just learning, Chay...

Timestamped segments:
  [0.0s - 3.2s] So we pay for that, they didn't, you know, there are some people who said,
  [3.2s - 6.6s] it's on Kren the Benzo instead of Quotan and musical,
  [6.6s - 8.6s] or there's not a story of what you're interested in just learning,
  [8.6s - 9.9s] Chayat, that's the...


In [4]:
print("=" * 65)
print("Experiment 3: Model Size Comparison")
print("=" * 65)
print()

if test_audio:
    model_sizes = ["tiny", "base", "small"]
    results = {}
    
    for size in model_sizes:
        print(f"\n--- Testing model: {size} ---")
        result = transcribe_with_faster_whisper(
            test_audio, model_size=size, device="cpu", compute_type="int8"
        )
        results[size] = result
    
    print("\n\n--- Speed Comparison ---")
    for size, res in results.items():
        print(f"  {size:>8}: {res['elapsed_seconds']:.2f}s")
    
    print("\n--- Transcription Comparison ---")
    for size, res in results.items():
        print(f"  {size:>8}: {res['text'][:100]}...")
else:
    print("No test audio available.")

Experiment 3: Model Size Comparison


--- Testing model: tiny ---
FASTER-WHISPER ASR (model: tiny, cpu/int8)
Audio: /home/conardd/working_space/Homework3-Submission/test_data/audio/sample-1.mp3
  Language: en (prob: 0.866)
  Segments: 4
  Transcribed in 0.8s
  Text: the way he pays a bat, they didn't, you know, that he's some people's sector son. Kren, the benzo, instead of court and a musical or just not story of which you're interested in just turning child, th...

--- Testing model: base ---
FASTER-WHISPER ASR (model: base, cpu/int8)
Audio: /home/conardd/working_space/Homework3-Submission/test_data/audio/sample-1.mp3
  Language: en (prob: 0.687)
  Segments: 6
  Transcribed in 4.8s
  Text: We pay for that, they don't, you know, there are some people, some people, some people, some, they're in the band, so instead of quoting a musical, or just not to hurry up, which are interesting, it's...

--- Testing model: small ---
FASTER-WHISPER ASR (model: small, cpu/int8)
Audio: /home/conardd/

In [5]:
# TODO 1: Analyze ASR for pretraining data
#
# Use the LLM to discuss how ASR fits into a pretraining pipeline.

print("=" * 65)
print("TODO 1: ASR in Pretraining Pipelines")
print("=" * 65)
print()

start = time.time()
response = client.generate(
    prompt="""Explain how Automatic Speech Recognition (ASR) is used to build LLM pretraining datasets.

Cover:
1. What audio sources are typically transcribed (podcasts, lectures, YouTube)?
2. What quality issues arise from ASR transcriptions vs written text?
3. How do modern models like Whisper v3 turbo (2025) compare to earlier ASR?
4. What post-processing is needed before ASR text goes into a training dataset?

Be practical and give specific examples.""",
    system="You are an expert in building LLM pretraining data pipelines.",
    max_tokens=500,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")

todo1_reflection = """
[YOUR REFLECTION HERE]
ASR transcription quality is worse than web-scraped text
FASTER-WHISPER seems better.
Definitely, quality of ASR transcriptions.
- How does ASR transcription quality compare to web-scraped text?
- Which model size would you choose for transcribing 1000 hours of podcasts?
- What are the ethical considerations of transcribing public audio content?
"""

print()
print(todo1_reflection)

TODO 1: ASR in Pretraining Pipelines

Response in 155.0s
Model: granite4:latest
Tokens: 125 in, 500 out
Stop reason: complete
Automatic Speech Recognition (ASR) plays a crucial role in building pretraining datasets for Large Language Models (LLMs). Here's how it works:

1. Audio Sources:
   - Podcasts: Many podcasts are transcribed using ASR, providing a large amount of conversational and informal text data.
   - Lectures: Academic lectures from universities or conferences can be transcribed to capture formal spoken content.
   - YouTube Videos: Millions of YouTube videos contain audio that can be transcribed, covering various topics like tutorials, interviews, news broadcasts, etc.

2. Quality Issues:
   - Noise Artifacts: ASR systems may introduce noise artifacts, punctuation errors, or misrecognized words, especially in noisy environments or when the speaker has a strong accent.
   - Lack of Context: ASR often struggles with context-dependent language, such as sarcasm, idioms, or do

In [12]:
# TODO 2: Transcribe your own audio (optional)
#
# Record a 15-30 second audio clip or find one online.
# Transcribe it and evaluate the quality.

print("=" * 65)
print("TODO 2: Custom Audio Transcription (Optional)")
print("=" * 65)
print()
class3_audio = os.path.join(parent_dir, '..', 'MLE_in_Gen_AI-Course', 'Class3', 'test_data', 'audio', 'sample-4.mp3')

my_audio = "../src/test_data/audio/sample-4.mp3"
my_result = transcribe_with_faster_whisper(class3_audio, model_size="small")
print(my_result)
todo2_reflection = """
[YOUR REFLECTION HERE]
I used sample-3, it loooks good, but there are some errors in the transcription, maybe because of the background noise.
- What audio did you transcribe?
- How accurate was the transcription?
- What would you change for better results (model size, preprocessing)?
"""

print(todo2_reflection)

TODO 2: Custom Audio Transcription (Optional)

FASTER-WHISPER ASR (model: small, cpu/int8)
Audio: /home/conardd/working_space/Homework3-Submission/../MLE_in_Gen_AI-Course/Class3/test_data/audio/sample-4.mp3
  Language: en (prob: 0.878)
  Segments: 2
  Transcribed in 3.6s
  Text: That's everything, never be is David nighting either down and there's that he said the post edition today that he start the good tending basis the false spot and now to...
{'text': "That's everything, never be is David nighting either down and there's that he said the post edition today that he start the good tending basis the false spot and now to", 'language': 'en', 'language_probability': 0.878, 'segments': [{'start': 0.0, 'end': 4.36, 'text': "That's everything, never be is David nighting either down and there's that he said the"}, {'start': 4.36, 'end': 9.84, 'text': 'post edition today that he start the good tending basis the false spot and now to'}], 'method': 'faster-whisper-small', 'elapsed_seconds': 3

---

## Summary & Reflection

In [13]:
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1 not completed yet]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2 not completed yet]'

full_reflection = f"""
### Part 1 - ASR in Pretraining

{_todo1}

---

### Part 2 - Custom Transcription

{_todo2}
"""

reflection_file = append_to_reflection(
    notebook="04",
    section_title="Speech Recognition (ASR)",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ../outputs/homework_reflection.md

API COST REPORT
Total API calls:     1
Total input tokens:  125
Total output tokens: 500
Total cost:          $0.0079

Last 1 calls:
  1. [12:21:53] granite4:latest -- 125in/500out -- $0.0079


## Notebook 04 Complete!

**What you accomplished:**
- Transcribed audio with OpenAI Whisper and faster-whisper
- Compared model sizes for speed vs accuracy
- Explored how ASR fits into pretraining pipelines

**Key concepts:**
- faster-whisper provides 4x speedup with CTranslate2 backend
- Whisper v3 turbo (2025) offers best speed/accuracy trade-off
- ASR text needs post-processing before use in pretraining

**Next:** Open **Notebook 05: Data Cleaning Pipeline**